# 04 - Real vs Synthetic Data Comparison: Validatation

So far, we have cleaned real ATP games data and got familiar with it (notebook 00). We then used Zermelo's algorithm to extract the strengths of players and get their true strengths for each year (notebook 01). Based on this truth, we calibrated the mathematical model to get parameters for creating synthetic data (maximum strength distribution, aging curves, retirement process, ...) (notebook 02). We ran an "empty" simulation (no tournaments) to validate the number of active players in the long term (stable number of active players, good potential distribution,...) (notebook 03). 

Now that we have everything we need for the tournaments simulation (with the functions contained in the `tournaments.py`), we can simulate the ATP circuit and compare the different rankings metrics (ATP points, Zermelo's strengths, PageRank scores, In-Degree Scores) with the hidden truth of our simulated data. We just verify that the real data look similar to the synthetic data.

## Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# for autoreload the modifications of the functions in the simulation.py file
%load_ext autoreload
%autoreload 2

In [ ]:
# importe the functions to run the full simulations
from tournaments import run_full_tournaments, prepare_tournament_schedule_points
from metrics import get_all_rankings, get_correlations, get_fraction_of_N_players

In [ ]:
# importing the parameters stored in the .json file
import json

config_path = "../config/simulation_params.json"

with open(config_path, "r") as f:
    config_params = json.load(f)

In [ ]:
# get the tournaments schedule and points 
tournament_points_path = "../config/rules/tournament_properties.txt"
tournament_schedule_path = "../config/rules/tournament_schedule.txt"

tournament_schedule_final, tournaments_points = prepare_tournament_schedule_points(tournament_points_path, tournament_schedule_path)

tournament_schedule_final.to_csv("../data/processed/tournaments_schedule.csv", index=False)
tournaments_points.to_csv("../data/processed/tournament_points.csv", index=False)

## Warm Up Years Needed

Before doing full simulations, we need to check how many warm-up years we need to have a stabilised system, with a good distribution of the ATP points (because at first the ATP points are distributed either at random or based on the initial strengths). The objective is to show after how many years the system has "forgotten" the initial conditions. 

In [ ]:
# Simulation 1 : Initialisation with ATP points proportional to the strength
warm_up_games, warm_up_rankings = run_full_tournaments(years=20, 
                                                       config_params=config_params, 
                                                       tournaments_schedule=tournament_schedule_final, 
                                                       tournaments_points=tournaments_points, 
                                                       seeding=True, random_initial_points=False)

warm_up_all_rankings = get_all_rankings(warm_up_games, warm_up_rankings, warm_up_years=0)
warm_up_correlations = get_correlations(warm_up_all_rankings, warm_up_years=0, return_all_years=True)

# Simulation 2 : Initialisation with random ATP points (between 0 and 2000)
random_initial_points_games, random_initial_points_rankings = run_full_tournaments(years=20, 
                                                                                   config_params=config_params, 
                                                                                   tournaments_schedule=tournament_schedule_final, 
                                                                                   tournaments_points=tournaments_points, 
                                                                                   seeding=True, random_initial_points=True)

random_initial_points_all_rankings = get_all_rankings(random_initial_points_games, random_initial_points_rankings, warm_up_years=0)
random_initial_points_correlations = get_correlations(random_initial_points_all_rankings, warm_up_years=0, return_all_years=True)

Here, we initialise all players with completely random ATP points (between 0 and 2000). The burn-in period will be determined as the number of years it takes for the ranking metrics to recover and reach stable balues.

In [ ]:
# visualize the convergence of the correlations
plt.figure(figsize=(20, 8))
sns.set_style("ticks") 
plt.rcParams.update({"font.family": "serif"})

plt.subplot(1,2,1)

plt.plot(warm_up_correlations["year"], warm_up_correlations["ATP_points"], label="ATP Points", linewidth=2, marker="o", markersize=8)
# plt.plot(warm_up_correlations["year"], warm_up_correlations["zermelo_strength"], label="Zermelo Strength", linewidth=2, marker="o", markersize=8)
# plt.plot(warm_up_correlations["year"], warm_up_correlations["pagerank_score"], label="PageRank Score", linewidth=2, marker="o", markersize=8)
# plt.plot(warm_up_correlations["year"], warm_up_correlations["in_degree"], label="In-Degree", linewidth=2, marker="o", markersize=8)

plt.title("Initial Points Proportional to Hidden Truth", fontsize=20, weight="bold", pad=15)
plt.xlabel("Simulation Year", fontsize=16)
plt.ylabel("Kendall's Tau Correlation", fontsize=16)
plt.xticks(np.arange(warm_up_correlations["year"].min(), warm_up_correlations["year"].max()+1, 3))
plt.legend(title="Ranking Metrics", fontsize=12, title_fontsize=13)
plt.grid(visible=True, which="major", axis="both", linewidth=0.5, alpha=0.5)


plt.subplot(1,2,2)

plt.plot(random_initial_points_correlations["year"], random_initial_points_correlations["ATP_points"], label="ATP Points", linewidth=2, marker="o", markersize=8)
# plt.plot(random_initial_points_correlations["year"], random_initial_points_correlations["zermelo_strength"], label="Zermelo Strength", linewidth=2, marker="o", markersize=8)
# plt.plot(random_initial_points_correlations["year"], random_initial_points_correlations["pagerank_score"], label="PageRank Score", linewidth=2, marker="o", markersize=8)
# plt.plot(random_initial_points_correlations["year"], random_initial_points_correlations["in_degree"], label="In-Degree", linewidth=2, marker="o", markersize=8)

plt.title("Random Initial Points", fontsize=20, weight="bold", pad=15)
plt.xlabel("Simulation Year", fontsize=16)
plt.xticks(np.arange(random_initial_points_correlations["year"].min(), random_initial_points_correlations["year"].max()+1, 3))

# plt.suptitle("Convergence of Metrics (Correlation: Metric vs. Hidden Truth)", fontsize=25, weight="bold")

plt.legend(title="Ranking Metrics", fontsize=12, title_fontsize=13)

plt.grid(visible=True, which="major", axis="both", linewidth=0.5, alpha=0.5)
sns.despine()
plt.tight_layout()
plt.show()

From this graph, we can set the burn-in period to **10 years**. After this burn-in period, all metrics converge to stable values, with the ATP points being the closest to the hidden truth. With random initial points, the initial values were lower than the stabilised values. Conversely, with the initial points proportional to the initial strengths, the initial values were higher than the stabilised values. In both cases, the system converges to the same stable values after around 10 years. This is a good sign that the system has "forgotten" the initial conditions.

## Generation of Synthetic Data vs Real Data

In [ ]:
# 1. REAL ATP DATA
ATP_data = pd.read_csv("../data/processed/all_matches_1991-2024.csv")
players_info = pd.read_csv("../data/processed/players_stats.csv")

# select ten years of data
ATP_start_year, ATP_end_year = 2010, 2019
ATP_10_years_data = ATP_data[(ATP_data["year"]>=ATP_start_year) & (ATP_data["year"]<=ATP_end_year)].copy()

# change the tourney level to match the categories of the synthetic data, and keep only the relevant categories (G, M, A, C, S)
ATP_10_years_data["tourney_level"] = ATP_10_years_data["tourney_level"].replace({"15": "S", "25": "S"})
ATP_10_years_data = ATP_10_years_data[ATP_10_years_data["tourney_level"].isin(["G", "M", "A", "C", "S"])]

# get the data of winners and losers, and concatenate them to get the data of all players
ATP_winners_data = ATP_10_years_data[["winner_id", "winner_rank", "winner_rank_points", "tourney_date", "year"]].copy().rename(columns={"winner_id": "player_id", "winner_rank": "rank", "winner_rank_points": "rank_points"})
ATP_losers_data = ATP_10_years_data[["loser_id", "loser_rank", "loser_rank_points", "tourney_date", "year"]].copy().rename(columns={"loser_id": "player_id", "loser_rank": "rank", "loser_rank_points": "rank_points"})
ATP_players_data = pd.concat([ATP_winners_data, ATP_losers_data], ignore_index=True)

# sort the values by tourney date
ATP_players_data = ATP_players_data.sort_values("tourney_date").reset_index(drop=True)

# keep only the last game of each player in each year (to get the ranking at the end of the year)
ATP_players_data = ATP_players_data.drop_duplicates(subset=["year", "player_id"], keep="last")

# get the age of the players from their birth year (and remove all players with missing birth year)
ATP_players_data = ATP_players_data.merge(players_info[["player_id", "birth_year"]], on="player_id", how="left")
ATP_players_data["age"] = ATP_players_data["year"] - ATP_players_data["birth_year"]
ATP_players_data = ATP_players_data.dropna(subset=["birth_year", "rank_points", "rank"]).reset_index(drop=True)

# sort the values by rank points (descending order) and then by year
ATP_players_data = ATP_players_data.sort_values(by=["year", "rank_points"], ascending=[True, False]).reset_index(drop=True)

# get the real ranking from the ATP data (based on the rank points)
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rank.html
ATP_players_data["real_rank"] = ATP_players_data.groupby("year")["rank_points"].rank(ascending=False, method="first").astype(int)

# count the tournaments played
all_real_matches = pd.concat([
    ATP_10_years_data[["winner_id", "tourney_id", "year"]].rename(columns={"winner_id": "player_id"}),
    ATP_10_years_data[["loser_id", "tourney_id", "year"]].rename(columns={"loser_id": "player_id"})
]).drop_duplicates()

tournaments_played_ATP = all_real_matches.groupby(["year", "player_id"]).size().reset_index(name="tournaments_played")

ATP_players_data = pd.merge(ATP_players_data, tournaments_played_ATP, on=["year", "player_id"], how="inner")

In [ ]:
# get synthetic data for 10 years (with a warm-up period of 10 years)
synthetic_games_20_years, synthetic_rankings_20_years = run_full_tournaments(years=20,
                                                                             config_params=config_params,
                                                                             tournaments_schedule=tournament_schedule_final,
                                                                             tournaments_points=tournaments_points,
                                                                             seeding=True,
                                                                             random_initial_points=True,
                                                                             track_week_ranks=True)

synthetic_all_rankings_10_years = get_all_rankings(synthetic_games_20_years, synthetic_rankings_20_years, warm_up_years=10)

To validate our model, we compare the 10 stable years of synthetic data (years 11 to 20, post burn-in) with 10 years of real ATP data (2010-2019). We focus on the distribution of points and the number of tournaments played.

## Validation of Demographics and System Dynamics

In [ ]:
# get the age distribution of players
plt.figure(figsize=(10, 6))

sns.kdeplot(synthetic_all_rankings_10_years["age"], fill=True, label="Synthetic Data", linewidth=2.5)
sns.kdeplot(ATP_players_data["age"], fill=True, label="Real Data (2010-2019)", linewidth=2.5)
plt.title("Age Distribution of Active Players", fontsize=18, weight="bold")
plt.xlabel("Age")
plt.ylabel("Density")
plt.legend()
sns.despine()
plt.show()

## Validation of the Model: Synthetic vs Real Data

We can first plot the distribution of the distribution of the ATP points for the real and synthetic data:

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

min_ATP_points = 1

synthetic_ATP_points = synthetic_all_rankings_10_years[synthetic_all_rankings_10_years["ATP_points"]>=min_ATP_points]["ATP_points"]
real_ATP_points = ATP_players_data[ATP_players_data["rank_points"]>=min_ATP_points]["rank_points"]

sns.kdeplot(synthetic_ATP_points, log_scale=True, fill=True, label="Synthetic Data (Years 11-20)", alpha=0.3, linewidth=3)
sns.kdeplot(real_ATP_points, log_scale=True, fill=True, label=f"Real Data ({ATP_start_year}-{ATP_end_year})", alpha=0.3, linewidth=3)

plt.title(f"Distribution of ATP Points (min. {min_ATP_points})", fontsize=25, weight="bold", pad=15)
plt.xlabel("ATP Points (Log Scale)", fontsize=14)
plt.ylabel("Density", fontsize=14)
plt.xlim(min_ATP_points, None)
plt.legend(fontsize=14, frameon=False)

plt.grid(visible=True, which="major", axis="both", linewidth=0.5, alpha=0.3)
sns.despine()
plt.tight_layout()
plt.show()

The problem could come from the fact that in real data, players that lost a lot of games won't continue playing and retiring, while in the synthetic data, players that lost a lot of games can continue playing and accumulating points. Let's check it by looking at the distribution of the number of tournaments played per year for the real and synthetic data:

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# remove the players with no tournaments in the synthetic data (as for real data, they do no appear)
synthetic_number_of_tournaments = synthetic_all_rankings_10_years[synthetic_all_rankings_10_years["tournaments_played"]>0]["tournaments_played"]

sns.kdeplot(synthetic_number_of_tournaments, fill=True, label="Synthetic Data (Years 11-20)", alpha=0.3, linewidth=3)
sns.kdeplot(ATP_players_data["tournaments_played"], fill=True, label="Real Data (2010-2019)", alpha=0.3, linewidth=3)

plt.title(f"Distribution of the Number of Tournaments Played", fontsize=25, weight="bold", pad=15)
plt.xlabel("Number of Tournaments", fontsize=14)
plt.ylabel("Density", fontsize=14)
plt.legend(fontsize=14, frameon=False)

plt.grid(visible=True, which="major", axis="both", linewidth=0.5, alpha=0.3)
sns.despine()
plt.tight_layout()
plt.show()

The peak is caused by the update of fatigue, with players that cannot play more than 2 consecutive weeks, so that the maximum number of tournaments played per year is the number of weeks (47) times $2/3$, which corresponds to a maximum of 32 tournaments per year. This explains directly the difference in the ATP points distribution, with the synthetic data having more players with a lot of tournaments played and thus more points accumulated.

To understand why the tournament distributions differ, we must check if the simulation contains the same number of matches per tournament as the real data.

In [ ]:
# number of tournaments per year and level for real and synthetic data
real_ATP_tournaments = ATP_10_years_data.drop_duplicates(subset=["year", "tourney_id"]).copy()
real_tourneys_per_year = real_ATP_tournaments.groupby("year")["tourney_level"].value_counts().unstack().mean().round(1)

synth_schedule_levels = tournament_schedule_final["level"].replace({
    2000: "G", 1000: "M", 500: "A", 250: "A", 
    125: "C", 110: "C", 100: "C", 90: "C", 80: "C",
    20: "S", 10: "S"
})
synth_tourneys_per_year = synth_schedule_levels.value_counts().astype(float)


# number of matches per tournament for real and synthetic data

#real number of matches per tournament (mean by level)
real_matches_per_tourney = ATP_10_years_data.groupby(["tourney_level", "year", "tourney_id"]).size().groupby(level='tourney_level').mean().round(1)

# remove the first 10 years of synthetic data (warm-up period)
synthetic_games_10_years = synthetic_games_20_years[synthetic_games_20_years["year"] > 10].copy()
synthetic_games_10_years["tourney_level"] = synthetic_games_10_years["tournament_level"].replace({
    2000: "G", 1000: "M", 500: "A", 250: "A", 
    125: "C", 110: "C", 100: "C", 90: "C", 80: "C",
    20: "S", 10: "S"
})

synth_matches_per_tourney = synthetic_games_10_years.groupby(["tourney_level", "year", "tournament_id"]).size().groupby(level='tourney_level').mean().round(1)


check_table_nbr_tournaments_games = pd.DataFrame({
    "Real Tourneys/Year": real_tourneys_per_year,
    "Synth Tourneys/Year": synth_tourneys_per_year,
    "Real Matches/Tourney": real_matches_per_tourney,
    "Synth Matches/Tourney": synth_matches_per_tourney
}).reindex(["G", "M", "A", "C", "S"])

display(check_table_nbr_tournaments_games)

Note: I had to remove some tournaments such that the number of matches per tournament and the number of such tournaments in a year correspond to the real data!

In [ ]:
print("Rounds in 'S' (Futures) tournaments in real ATP data:")
display(list(ATP_10_years_data[ATP_10_years_data["tourney_level"] == "S"]["round"].unique()))

print("\nRounds in 'S' (Futures) tournaments in synthetic data:")
display(list(synthetic_games_10_years[synthetic_games_10_years["tourney_level"] == "S"]["round"].unique()))

(All qualifications have been removed for the 10 and 20 tournaments because they do not exist in real data.)

In [ ]:
synthetic_number_games_per_year = synthetic_games_10_years.groupby("year").size().mean()
ATP_number_games_per_year = ATP_10_years_data.groupby("year").size().mean()

print("Number of games per year in synthetic data (years 11-20):")
print(synthetic_number_games_per_year)
print("\nNumber of games per year in real ATP data (2010-2019):")
print(ATP_number_games_per_year)

As we are also interested in looking at the Top players, we can compare the distribution of the ATP points for the Top 100 players in the real and synthetic data:

In [ ]:
# ATP points of all players and of top
ATP_total_points = ATP_players_data.groupby("year")["rank_points"].sum()
ATP_top_100_points = ATP_players_data[ATP_players_data["real_rank"]<=100].groupby("year")["rank_points"].sum()
ATP_top_10_points = ATP_players_data[ATP_players_data["real_rank"]<=10].groupby("year")["rank_points"].sum()

# get the ATP ranking of simulated data
synthetic_all_rankings_10_years["ATP_rank"] = synthetic_all_rankings_10_years.groupby("year")["ATP_points"].rank(ascending=False, method="first")

synthetic_all_points = synthetic_all_rankings_10_years.groupby("year")["ATP_points"].sum()
synthetic_top_100_points = synthetic_all_rankings_10_years[synthetic_all_rankings_10_years["ATP_rank"]<=100].groupby("year")["ATP_points"].sum()
synthetic_top_10_points = synthetic_all_rankings_10_years[synthetic_all_rankings_10_years["ATP_rank"]<=10].groupby("year")["ATP_points"].sum()

ATP_ratio_points = (ATP_top_100_points / ATP_total_points).mean()
synthetic_ratio_points = (synthetic_top_100_points / synthetic_all_points).mean()

ATP_ratio_std = (ATP_top_100_points / ATP_total_points).std()
synthetic_ratio_std = (synthetic_top_100_points / synthetic_all_points).std()

print("Fraction of ATP points held by the Top 100 players:")
print(f"- ATP real data: {ATP_ratio_points*100:.2f}% ± {ATP_ratio_std*100:.2f}%")
print(f"- ATP simulated data: {synthetic_ratio_points*100:.2f}% ± {synthetic_ratio_std*100:.2f}%")

ATP_top_10_ratio = (ATP_top_10_points / ATP_top_100_points).mean()
synthetic_top_10_ratio = (synthetic_top_10_points / synthetic_top_100_points).mean()

ATP_top_10_std = (ATP_top_10_points / ATP_top_100_points).std()
synthetic_top_10_std = (synthetic_top_10_points / synthetic_top_100_points).std()

print("\nFraction of ATP points held by the Top 10 players compared to Top 100 Players:")

print(f"- ATP real data: {ATP_top_10_ratio*100:.2f}% ± {ATP_top_10_std*100:.2f}%")
print(f"- ATP simulated data: {synthetic_top_10_ratio*100:.2f}% ± {synthetic_top_10_std*100:.2f}%")

In [ ]:
ranks = range(1,101,1)

fraction_rank_ATP_points = []
fraction_rank_ATP_points_std = []
fraction_rank_synthetic_points = []
fraction_rank_synthetic_points_std = []

for rank in ranks:

    real_top_rank_points = ATP_players_data[ATP_players_data["real_rank"] <= rank].groupby("year")["rank_points"].sum()
    synthetic_top_rank_points = synthetic_all_rankings_10_years[synthetic_all_rankings_10_years["ATP_rank"] <= rank].groupby("year")["ATP_points"].sum()

    fraction_rank_ATP_points.append((real_top_rank_points / ATP_top_100_points).mean())
    fraction_rank_ATP_points_std.append((real_top_rank_points / ATP_top_100_points).std())
    fraction_rank_synthetic_points.append((synthetic_top_rank_points / synthetic_top_100_points).mean())
    fraction_rank_synthetic_points_std.append((synthetic_top_rank_points / synthetic_top_100_points).std())

In [ ]:
plt.figure(figsize=(10, 6))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

plt.errorbar(ranks, fraction_rank_synthetic_points, yerr=fraction_rank_synthetic_points_std, label="Synthetic Data", linewidth=2, elinewidth=1, capsize=3)
plt.errorbar(ranks, fraction_rank_ATP_points, yerr=fraction_rank_ATP_points_std, label="Real Data (2010-2019)", linewidth=2, elinewidth=1, capsize=3)


plt.title("Distribution of ATP Points (Top 100 Players)", fontsize=16, weight="bold", pad=15)
plt.xlabel("Player Rank", fontsize=14)
plt.ylabel("Fraction of Top 100 Points", fontsize=14)
plt.legend(fontsize=12)
plt.xlim(1,100)
plt.ylim(0, 1)

plt.grid(visible=True, which="major", linestyle="--", alpha=0.3)
sns.despine()
plt.tight_layout()
plt.show()

The curve looks similar, with a slight shift but nothing too worrying.



In [ ]:
# real ATP ranks
ATP_win_ranks = ATP_10_years_data[["tourney_level", "winner_rank"]].rename(columns={"winner_rank": "rank"})
ATP_loss_ranks = ATP_10_years_data[["tourney_level", "loser_rank"]].rename(columns={"loser_rank": "rank"}) 
ATP_all_ranks = pd.concat([ATP_win_ranks, ATP_loss_ranks], ignore_index=True).dropna()

# replace the tourney levels 15 and 25 by 'S' (for small tournaments)
ATP_all_ranks["tourney_level"] = ATP_all_ranks["tourney_level"].replace({"15": "S", "25": "S"})
ATP_ranks = ATP_all_ranks[ATP_all_ranks["tourney_level"].isin(["G", "M", "A", "C", "S"])].copy()
ATP_ranks["data"] = "real"

# synthetic ranks
synthetic_win_ranks = synthetic_games_10_years[["tournament_level", "winner_rank"]].rename(columns={"winner_rank": "rank", "tournament_level": "tourney_level"})
synthetic_loss_ranks = synthetic_games_10_years[["tournament_level", "loser_rank"]].rename(columns={"loser_rank": "rank", "tournament_level": "tourney_level"})
synthetic_all_ranks = pd.concat([synthetic_win_ranks, synthetic_loss_ranks], ignore_index=True).dropna()
synthetic_all_ranks["tourney_level"] = synthetic_all_ranks["tourney_level"].replace({2000: "G", 1000: "M", 
                                                                                           500: "A", 250: "A", 
                                                                                           125: "C", 110: "C", 100: "C", 90: "C", 80: "C",
                                                                                            20: "S", 10: "S"})
synthetic_all_ranks["data"] = "simulated" 

all_ranks = pd.concat([ATP_ranks, synthetic_all_ranks], ignore_index=True)

In [ ]:
plt.figure(figsize=(16, 8))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

sns.boxplot(x="tourney_level", y="rank", hue="data", data=all_ranks, order=["G", "M", "A", "C", "S"], showfliers=False)

plt.title("Player Entry Rank by Tournament Level", fontsize=20, weight="bold", pad=15)
plt.xlabel("Tournament Level", fontsize=14)
plt.ylabel("Player ATP Rank", fontsize=14)

plt.gca().invert_yaxis() # get the rank in reverse order
plt.ylim(1000, 1)

plt.legend(fontsize=12, loc='lower left')
plt.grid(visible=True, which="major", axis="y", linestyle="--", alpha=0.3)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Fonction pour créer les groupes de classement
def assign_tier(rank):
    if rank <= 100: return "Top 100"
    if rank <= 300: return "101-300"
    if rank <= 600: return "301-600"
    if rank <= 1000: return "601-1000"
    return "1001+"

# --- LE FILTRE CRUCIAL EST ICI ---
# On ne garde que les joueurs actifs (tournaments_played > 0)

# 1. Préparation des données synthétiques
synth_clean = synthetic_all_rankings_10_years[synthetic_all_rankings_10_years["tournaments_played"] > 0]
synth_stratified = synth_clean.copy()
synth_stratified["Tier"] = synth_stratified["ATP_rank"].apply(assign_tier) # Attention : vérifie si c'est "ATP_rank" ou juste "rank" dans ton DF
synth_stratified["Dataset"] = "Synthetic"

# 2. Préparation des données réelles (ATP 2010-2019)
real_clean = ATP_players_data[ATP_players_data["tournaments_played"] > 0]
real_stratified = real_clean.copy()
real_stratified["Tier"] = real_stratified["real_rank"].apply(assign_tier)
real_stratified["Dataset"] = "Real (ATP)"

# 3. Combinaison et Affichage
combined_tiers = pd.concat([
    synth_stratified[["Tier", "tournaments_played", "Dataset"]],
    real_stratified[["Tier", "tournaments_played", "Dataset"]]
])

plt.figure(figsize=(14, 7))
sns.set_style("ticks")

labels_order = ["Top 100", "101-300", "301-600", "601-1000", "1001+"]

sns.boxplot(data=combined_tiers, x="Tier", y="tournaments_played", hue="Dataset", 
            order=labels_order, 
            showfliers=False, palette={"Real (ATP)": "#1f77b4", "Synthetic": "#ff7f0e"})

plt.ylabel("Tournaments Played", fontsize=13)
plt.xlabel("Player Ranking Tier", fontsize=13)
plt.grid(axis='y', linestyle='--', alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()